<a href="https://colab.research.google.com/github/MilagrosBrucellaria/preentrega-7-sql-features/blob/main/preentrega_7_sql_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preentrega 7 - Ingeniería de Features con SQL Analítico Avanzado

## Objetivo

El objetivo de esta práctica es transformar datos transaccionales de e-commerce en un dataset de features por cliente utilizando SQL analítico avanzado.

Se generarán variables de comportamiento como:

- **Recency:** días desde la última compra.
- **Frequency:** cantidad de pedidos únicos.
- **Monetary:** gasto total acumulado.
- **Trend:** diferencia entre el gasto del último mes y el promedio histórico.

Estas variables permiten resumir el comportamiento de cada cliente y pueden utilizarse posteriormente en modelos de Machine Learning orientados, por ejemplo, a la predicción de abandono de clientes (churn).

Para realizar el análisis se utilizará **DuckDB**, que permite ejecutar consultas SQL directamente desde Python.

In [1]:
!pip install duckdb==1.4.1 openpyxl==3.1.5

## 1. Importación de librerías

Se importan las librerías necesarias para trabajar con los datos.

- **pandas:** permite leer y visualizar el dataset.
- **duckdb:** permite ejecutar consultas SQL analíticas.
- **google.colab.files:** permite cargar y descargar archivos desde Google Colab.

In [2]:
import pandas as pd
import duckdb

from google.colab import files

print("Pandas:", pd.__version__)
print("DuckDB:", duckdb.__version__)

Pandas: 2.2.3
DuckDB: 1.4.1


## 2. Fuente y carga del dataset

Se utilizará el dataset **Online Retail** del UCI Machine Learning Repository.

Fuente oficial:

https://archive.ics.uci.edu/dataset/352/online+retail

El dataset contiene transacciones de una tienda online del Reino Unido realizadas entre diciembre de 2010 y diciembre de 2011.

A continuación se carga el archivo original `Online Retail.xlsx`.

In [4]:
from google.colab import files

uploaded = files.upload()

Saving Online Retail.xlsx to Online Retail.xlsx


In [5]:
df = pd.read_excel("Online Retail.xlsx")

print("Cantidad de filas:", len(df))
print("Cantidad de columnas:", len(df.columns))

df.head()

Cantidad de filas: 541909
Cantidad de columnas: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Exploración inicial

Antes de construir las features, se revisa la estructura del dataset,
los tipos de datos y la existencia de valores faltantes.

Esta revisión permite identificar qué variables necesitan limpieza antes
de realizar los cálculos mediante SQL.

In [6]:
print("Columnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

Columnas:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Tipos de datos:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Valores nulos:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


## Preparación del entorno SQL

Para realizar la ingeniería de features se utilizará DuckDB.

El DataFrame se registra con el nombre `online_retail`, lo que permite
consultar los datos directamente mediante SQL sin necesidad de crear
previamente una base de datos física.

In [7]:
# Crear conexión con DuckDB
con = duckdb.connect()

# Registrar el DataFrame como una tabla para utilizarlo desde SQL
con.register("online_retail", df)

print("DataFrame registrado correctamente en DuckDB.")

DataFrame registrado correctamente en DuckDB.


In [8]:
resultado = con.execute("""
    SELECT COUNT(*) AS cantidad_registros
    FROM online_retail
""").df()

resultado

,cantidad_registros
0,541909


## Ingeniería de Features con SQL

A partir de las transacciones válidas se construirá un dataset con una fila por cliente.

Se generarán las siguientes variables:

- **Recency:** días desde la última compra.
- **Frequency:** cantidad de pedidos únicos.
- **Monetary:** gasto total acumulado.
- **Trend:** diferencia entre el gasto del último mes y el promedio histórico.

También se utilizarán funciones de ventana para analizar el comportamiento de cada cliente a lo largo del tiempo.

In [9]:
query = """

WITH valid_transactions AS (

    -- 1. Limpiamos las transacciones
    SELECT
        CAST(CustomerID AS VARCHAR) AS customer_id,
        CAST(InvoiceNo AS VARCHAR) AS invoice_no,
        CAST(InvoiceDate AS TIMESTAMP) AS invoice_date,
        CAST(Quantity AS INTEGER) AS quantity,
        CAST(UnitPrice AS DOUBLE) AS unit_price,
        CAST(Quantity AS DOUBLE) * CAST(UnitPrice AS DOUBLE) AS amount

    FROM online_retail

    WHERE CustomerID IS NOT NULL
      AND Quantity > 0
      AND UnitPrice > 0
      AND UPPER(CAST(InvoiceNo AS VARCHAR)) NOT LIKE 'C%'
),

reference_date AS (

    -- 2. Definimos una fecha de referencia
    SELECT
        MAX(CAST(invoice_date AS DATE)) + INTERVAL 1 DAY AS analysis_date
    FROM valid_transactions
),

customer_rfm AS (

    -- 3. Calculamos Recency, Frequency y Monetary
    SELECT
        customer_id,

        DATE_DIFF(
            'day',
            MAX(CAST(invoice_date AS DATE)),
            (SELECT analysis_date FROM reference_date)
        ) AS recency_days,

        COUNT(DISTINCT invoice_no) AS frequency_orders,

        ROUND(SUM(amount), 2) AS monetary_total

    FROM valid_transactions

    GROUP BY customer_id
),

monthly_spending AS (

    -- 4. Calculamos cuánto gastó cada cliente por mes
    SELECT
        customer_id,
        DATE_TRUNC('month', invoice_date) AS purchase_month,
        SUM(amount) AS monthly_amount

    FROM valid_transactions

    GROUP BY
        customer_id,
        DATE_TRUNC('month', invoice_date)
),

monthly_features AS (

    -- 5. Aplicamos funciones de ventana
    SELECT
        customer_id,
        purchase_month,
        monthly_amount,

        -- Ventana 1: identifica el último mes
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY purchase_month DESC
        ) AS month_position,

        -- Ventana 2: promedio histórico
        AVG(monthly_amount) OVER (
            PARTITION BY customer_id
            ORDER BY purchase_month
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS historical_monthly_average,

        -- Ventana 3: gasto acumulado
        SUM(monthly_amount) OVER (
            PARTITION BY customer_id
            ORDER BY purchase_month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS accumulated_spending

    FROM monthly_spending
),

latest_month AS (

    -- 6. Nos quedamos con el último mes de cada cliente
    SELECT
        customer_id,
        monthly_amount AS last_month_spending,

        COALESCE(
            historical_monthly_average,
            0
        ) AS historical_monthly_average,

        accumulated_spending

    FROM monthly_features

    WHERE month_position = 1
)

-- 7. Construimos el dataset final
SELECT
    rfm.customer_id,

    COALESCE(rfm.recency_days, 0) AS recency_days,

    COALESCE(rfm.frequency_orders, 0) AS frequency_orders,

    COALESCE(rfm.monetary_total, 0) AS monetary_total,

    ROUND(
        COALESCE(lm.last_month_spending, 0),
        2
    ) AS last_month_spending,

    ROUND(
        COALESCE(lm.historical_monthly_average, 0),
        2
    ) AS historical_monthly_average,

    ROUND(
        COALESCE(lm.last_month_spending, 0)
        - COALESCE(lm.historical_monthly_average, 0),
        2
    ) AS spending_trend,

    ROUND(
        COALESCE(lm.accumulated_spending, 0),
        2
    ) AS accumulated_spending

FROM customer_rfm AS rfm

LEFT JOIN latest_month AS lm
    ON rfm.customer_id = lm.customer_id

ORDER BY rfm.customer_id

"""

# Ejecutamos la consulta
features = con.execute(query).df()

# Mostramos el resultado
features.head(10)

,customer_id,recency_days,frequency_orders,monetary_total,last_month_spending,historical_monthly_average,spending_trend,accumulated_spending
0,12346.0,326,1,77183.60,77183.60,0.00,77183.60,77183.60
1,12347.0,3,7,4310.00,224.82,680.86,-456.04,4310.00
2,12348.0,76,4,1797.24,310.00,495.75,-185.75,1797.24
3,12349.0,19,1,1757.55,1757.55,0.00,1757.55,1757.55
4,12350.0,311,1,334.40,334.40,0.00,334.40,334.40
5,12352.0,37,8,2506.04,311.73,731.44,-419.71,2506.04
6,12353.0,205,1,89.00,89.00,0.00,89.00,89.00
7,12354.0,233,1,1079.40,1079.40,0.00,1079.40,1079.40
8,12355.0,215,1,459.40,459.40,0.00,459.40,459.40
9,12356.0,23,3,2811.43,58.35,1376.54,-1318.19,2811.43


## Funciones de ventana utilizadas

Para analizar el comportamiento temporal de cada cliente se utilizaron funciones de ventana con `OVER` y `PARTITION BY`.

Se implementaron tres funciones:

1. **ROW_NUMBER() OVER:** permite identificar el último mes de actividad de cada cliente.
2. **AVG() OVER:** calcula el promedio de gasto mensual histórico anterior al último mes.
3. **SUM() OVER:** calcula el gasto acumulado del cliente.

El uso de `PARTITION BY customer_id` permite realizar estos cálculos de forma independiente para cada cliente.

## 7. Validación del dataset de features

Una vez ejecutada la consulta SQL, se verifica la estructura del dataset resultante.

El objetivo es comprobar que exista una única fila por cliente y que las variables generadas sean consistentes.

In [10]:
print("Cantidad de clientes:", len(features))

print("\nColumnas generadas:")
print(features.columns.tolist())

print("\nPrimeros registros:")
display(features.head(10))

Cantidad de clientes: 4338

Columnas generadas:
['customer_id', 'recency_days', 'frequency_orders', 'monetary_total', 'last_month_spending', 'historical_monthly_average', 'spending_trend', 'accumulated_spending']

Primeros registros:


,customer_id,recency_days,frequency_orders,monetary_total,last_month_spending,historical_monthly_average,spending_trend,accumulated_spending
0,12346.0,326,1,77183.60,77183.60,0.00,77183.60,77183.60
1,12347.0,3,7,4310.00,224.82,680.86,-456.04,4310.00
2,12348.0,76,4,1797.24,310.00,495.75,-185.75,1797.24
3,12349.0,19,1,1757.55,1757.55,0.00,1757.55,1757.55
4,12350.0,311,1,334.40,334.40,0.00,334.40,334.40
5,12352.0,37,8,2506.04,311.73,731.44,-419.71,2506.04
6,12353.0,205,1,89.00,89.00,0.00,89.00,89.00
7,12354.0,233,1,1079.40,1079.40,0.00,1079.40,1079.40
8,12355.0,215,1,459.40,459.40,0.00,459.40,459.40
9,12356.0,23,3,2811.43,58.35,1376.54,-1318.19,2811.43


In [11]:
print("Valores nulos por columna:")
print(features.isnull().sum())

Valores nulos por columna:
customer_id                   0
recency_days                  0
frequency_orders              0
monetary_total                0
last_month_spending           0
historical_monthly_average    0
spending_trend                0
accumulated_spending          0
dtype: int64


In [12]:
print("Clientes duplicados:")
print(features["customer_id"].duplicated().sum())

print("\nResumen estadístico:")
display(features.describe())

Clientes duplicados:
0

Resumen estadístico:


,recency_days,frequency_orders,monetary_total,last_month_spending,historical_monthly_average,spending_trend,accumulated_spending
count,4338.000000,4338.000000,4338.000000,4338.000000,4338.000000,4338.000000,4338.000000
mean,93.059474,4.272015,2054.266459,538.038970,340.063034,197.976150,2054.266459
std,100.012264,7.697998,8989.230441,2985.631562,895.295410,2969.051541,8989.230441
min,1.000000,1.000000,3.750000,3.350000,0.000000,-15982.000000,3.750000
25%,18.000000,1.000000,307.415000,170.170000,0.000000,-64.157500,307.415000
50%,51.000000,2.000000,674.485000,307.860000,195.945000,102.875000,674.485000
75%,142.750000,5.000000,1661.740000,500.210000,417.597500,300.880000,1661.740000
max,374.000000,209.000000,280206.020000,168469.600000,22998.400000,168466.700000,280206.020000


## Interpretación de las Features

Las variables generadas representan diferentes dimensiones del comportamiento de compra:

- **Recency:** indica hace cuántos días realizó su última compra el cliente. Una recencia alta puede indicar menor actividad reciente.

- **Frequency:** representa la cantidad de pedidos únicos realizados. Una mayor frecuencia puede asociarse con un cliente más activo.

- **Monetary:** representa el gasto total acumulado del cliente y permite medir su valor económico.

- **Last Month Spending:** representa el gasto realizado durante el último mes con actividad.

- **Historical Monthly Average:** representa el gasto mensual promedio anterior al último mes.

- **Spending Trend:** compara el gasto del último mes con el promedio histórico. Un valor negativo refleja una disminución del gasto reciente.

- **Accumulated Spending:** representa el gasto acumulado del cliente a lo largo del período analizado.

Estas variables pueden funcionar como señales relevantes para un modelo de churn, ya que permiten identificar cambios en la actividad y en los patrones de compra.

## Análisis de la variable Trend

La variable `spending_trend` se calcula como:

**Gasto del último mes - Promedio mensual histórico**

Su interpretación es:

- **Trend > 0:** el gasto reciente está por encima del promedio histórico.
- **Trend = 0:** no existe diferencia o no existe suficiente historia previa.
- **Trend < 0:** el gasto reciente está por debajo del comportamiento histórico.

Una tendencia negativa puede representar una señal de disminución de actividad y resultar útil para detectar posibles clientes con riesgo de churn.

## Generación del archivo SQL
La consulta principal utilizada para generar las features se exportará a un archivo llamado `features_clientes.sql`.

Este archivo contiene la lógica completa de preparación de las variables y será incluido posteriormente en el repositorio de GitHub.

In [13]:
sql_file = """-- ============================================================
-- PREENTREGA 7
-- Ingeniería de Features con SQL Analítico Avanzado
-- Features: Recency, Frequency, Monetary y Trend
-- ============================================================

""" + query

with open("features_clientes.sql", "w", encoding="utf-8") as archivo:
    archivo.write(sql_file)

print("✅ Archivo features_clientes.sql creado correctamente.")

✅ Archivo features_clientes.sql creado correctamente.


In [17]:
with open("features_clientes.sql", "r", encoding="utf-8") as archivo:
    print(archivo.read()[:1500])

-- ============================================================
-- PREENTREGA 7
-- Ingeniería de Features con SQL Analítico Avanzado
-- Features: Recency, Frequency, Monetary y Trend
-- ============================================================



WITH valid_transactions AS (

    -- 1. Limpiamos las transacciones
    SELECT
        CAST(CustomerID AS VARCHAR) AS customer_id,
        CAST(InvoiceNo AS VARCHAR) AS invoice_no,
        CAST(InvoiceDate AS TIMESTAMP) AS invoice_date,
        CAST(Quantity AS INTEGER) AS quantity,
        CAST(UnitPrice AS DOUBLE) AS unit_price,
        CAST(Quantity AS DOUBLE) * CAST(UnitPrice AS DOUBLE) AS amount

    FROM online_retail

    WHERE CustomerID IS NOT NULL
      AND Quantity > 0
      AND UnitPrice > 0
      AND UPPER(CAST(InvoiceNo AS VARCHAR)) NOT LIKE 'C%'
),

reference_date AS (

    -- 2. Definimos una fecha de referencia
    SELECT
        MAX(CAST(invoice_date AS DATE)) + INTERVAL 1 DAY AS analysis_date
    FROM valid_transactions
)

In [19]:
files.download("features_clientes.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Exportación del dataset de features

Como resultado adicional del proceso se exporta el dataset final en formato CSV.

Este archivo contiene una fila por cliente junto con las variables generadas mediante la consulta SQL.

El archivo de datos se utilizará como resultado del procesamiento, pero no será incluido en el repositorio de GitHub, ya que la carpeta de datos estará definida en `.gitignore`.

In [18]:
features.to_csv(
    "features_clientes.csv",
    index=False
)

print("Archivo features_clientes.csv generado correctamente.")

Archivo features_clientes.csv generado correctamente.


## Conclusión

En esta práctica se realizó un proceso de ingeniería de features sobre datos transaccionales de e-commerce utilizando SQL analítico avanzado.

Se construyeron las variables **Recency, Frequency, Monetary y Trend**, junto con métricas complementarias de comportamiento.

Para su construcción se utilizaron funciones de fecha, agregaciones, `GROUP BY`, filtros, tratamiento de valores nulos mediante `COALESCE` y funciones de ventana con `OVER` y `PARTITION BY`.

Además, se realizaron controles de valores nulos y clientes duplicados para verificar la consistencia del dataset generado.

El resultado final contiene una fila por cliente y representa un dataset preparado para ser utilizado posteriormente como entrada de un modelo de Machine Learning orientado a la predicción de churn.